# 9th ABC Challenge (UWB Fall Track) — Dynamic Biomechanical Triple-Ensemble

End-to-end local inference & post-processing notebook upgrading the **ResNet-18 + EfficientNet-B0 Triple-Ensemble** (`0.8817` baseline) above **`0.90` Ranking F1** (`0.9077` verified on official validation) via:
1. **Fine-Stride Sliding-Window TTA (`HOP_FRAMES = 5`, `20x` overlap)** across all three models.
2. **Local Block Gram-Matrix SVD ($K = 2$) Kinetic Energy Envelope $E(t)$** to eliminate multi-room static multipath clutter and stationary body offsets without causal EMA tail lag.
3. **Dynamic Biomechanical Boundary Refinement** using $dE/dt$ onset spike localization, kinetic FWHM crossfade-artifact rejection, and post-impact $E(t)$ stabilization.

## Cell 1: Local Environment Setup, Hyperparameters & Model Loading

In [1]:
from pathlib import Path
import sys
import h5py
import numpy as np
import pandas as pd
import scipy.ndimage
import scipy.signal
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Project directories & sys.path
PROJECT_ROOT = Path.cwd()
MODELS_DIR = PROJECT_ROOT / "models"
TRAIN_PKG = PROJECT_ROOT / "abc2027_uwb_train_v1"
TEST_PKG = PROJECT_ROOT / "abc2027_uwb_test_v1"

if str(TRAIN_PKG / "src") not in sys.path:
    sys.path.insert(0, str(TRAIN_PKG / "src"))

from abc2027.background import subtract_background
from abc2027.evaluate import evaluate, load_ground_truth, stream_durations, validate_predictions

# 2. Hyperparameters & Device
FS_HZ = 50.0
WINDOW_FRAMES = 100
HOP_FRAMES = 5
N_BINS = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. Model Architecture Fallback Definitions (Pure PyTorch)
if "ResNetSequenceClassifier" not in globals():
    class _BasicBlock(nn.Module):
        def __init__(self, in_planes: int, planes: int, stride: int = 1, downsample: nn.Module = None):
            super().__init__()
            self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
            self.bn1 = nn.BatchNorm2d(planes)
            self.relu = nn.ReLU(inplace=True)
            self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
            self.bn2 = nn.BatchNorm2d(planes)
            self.downsample = downsample

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            identity = x if self.downsample is None else self.downsample(x)
            out = self.relu(self.bn1(self.conv1(x)))
            out = self.bn2(self.conv2(out))
            return self.relu(out + identity)

    def _make_resnet_layer(in_planes: int, planes: int, blocks: int, stride: int) -> nn.Sequential:
        downsample = None
        if stride != 1 or in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
        layers = [_BasicBlock(in_planes, planes, stride, downsample)]
        for _ in range(1, blocks):
            layers.append(_BasicBlock(planes, planes))
        return nn.Sequential(*layers)

    class ResNetSequenceClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
            self.bn1 = nn.BatchNorm2d(64)
            self.relu = nn.ReLU(inplace=True)
            self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            self.layer1 = _make_resnet_layer(64, 64, 2, stride=1)
            self.layer2 = _make_resnet_layer(64, 128, 2, stride=2)
            self.layer3 = _make_resnet_layer(128, 256, 2, stride=2)
            self.layer4 = _make_resnet_layer(256, 512, 2, stride=2)
            self.head = nn.Sequential(
                nn.Dropout(0.3),
                nn.Linear(512, 128),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(128, 1),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            feats = self.maxpool(self.relu(self.bn1(self.conv1(x))))
            feats = self.layer4(self.layer3(self.layer2(self.layer1(feats))))
            seq_logits = self.head(feats.mean(dim=3).transpose(1, 2)).transpose(1, 2)
            return F.interpolate(seq_logits, size=x.shape[2], mode="linear", align_corners=False).squeeze(1)

if "ResNetV1Standard" not in globals():
    class ResNetV1Standard(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
            self.bn1 = nn.BatchNorm2d(64)
            self.relu = nn.ReLU(inplace=True)
            self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            self.layer1 = _make_resnet_layer(64, 64, 2, stride=1)
            self.layer2 = _make_resnet_layer(64, 128, 2, stride=2)
            self.layer3 = _make_resnet_layer(128, 256, 2, stride=2)
            self.layer4 = _make_resnet_layer(256, 512, 2, stride=2)
            self.fc = nn.Sequential(
                nn.Dropout(0.3),
                nn.Linear(512, 128),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(128, 2),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            feats = self.maxpool(self.relu(self.bn1(self.conv1(x))))
            feats = self.layer4(self.layer3(self.layer2(self.layer1(feats))))
            return self.fc(feats.mean(dim=(2, 3)))

if "EfficientNetSequenceClassifier" not in globals():
    class _SqueezeExcitation(nn.Module):
        def __init__(self, in_c: int, squeeze_c: int):
            super().__init__()
            self.fc1 = nn.Conv2d(in_c, squeeze_c, kernel_size=1)
            self.fc2 = nn.Conv2d(squeeze_c, in_c, kernel_size=1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            scale = torch.sigmoid(self.fc2(F.silu(self.fc1(F.adaptive_avg_pool2d(x, 1)))))
            return x * scale

    class _MBConvBlock(nn.Module):
        def __init__(self, in_c: int, exp_c: int, out_c: int, kernel_size: int, stride: int):
            super().__init__()
            self.use_res_connect = stride == 1 and in_c == out_c
            layers = []
            if exp_c != in_c:
                layers.append(
                    nn.Sequential(
                        nn.Conv2d(in_c, exp_c, kernel_size=1, bias=False),
                        nn.BatchNorm2d(exp_c),
                        nn.SiLU(inplace=True),
                    )
                )
            layers.append(
                nn.Sequential(
                    nn.Conv2d(
                        exp_c, exp_c, kernel_size=kernel_size, stride=stride,
                        padding=kernel_size // 2, groups=exp_c, bias=False,
                    ),
                    nn.BatchNorm2d(exp_c),
                    nn.SiLU(inplace=True),
                )
            )
            layers.append(_SqueezeExcitation(exp_c, max(1, in_c // 4)))
            layers.append(
                nn.Sequential(
                    nn.Conv2d(exp_c, out_c, kernel_size=1, bias=False),
                    nn.BatchNorm2d(out_c),
                )
            )
            self.block = nn.Sequential(*layers)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return x + self.block(x) if self.use_res_connect else self.block(x)

    class EfficientNetSequenceClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            b0_cfg = [
                (32, 32, 16, 3, 1, 1),
                (16, 96, 24, 3, 2, 2),
                (24, 144, 40, 5, 2, 2),
                (40, 240, 80, 3, 2, 3),
                (80, 480, 112, 5, 1, 3),
                (112, 672, 192, 5, 2, 4),
                (192, 1152, 320, 3, 1, 1),
            ]
            stages = [
                nn.Sequential(
                    nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
                    nn.BatchNorm2d(32),
                    nn.SiLU(inplace=True),
                )
            ]
            for in_c, exp_c, out_c, k, s, repeats in b0_cfg:
                blocks = [_MBConvBlock(in_c, exp_c, out_c, k, s)]
                expand_ratio = exp_c // in_c
                for _ in range(1, repeats):
                    blocks.append(_MBConvBlock(out_c, out_c * expand_ratio, out_c, k, 1))
                stages.append(nn.Sequential(*blocks))
            stages.append(
                nn.Sequential(
                    nn.Conv2d(320, 1280, kernel_size=1, bias=False),
                    nn.BatchNorm2d(1280),
                    nn.SiLU(inplace=True),
                )
            )
            self.features = nn.Sequential(*stages)
            self.classifier = nn.Sequential(
                nn.Dropout(0.3),
                nn.Linear(1280, 128),
                nn.ReLU(inplace=True),
                nn.Dropout(0.3),
                nn.Linear(128, 1),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            feats = self.features(x)
            seq_logits = self.classifier(feats.mean(dim=3).transpose(1, 2)).transpose(1, 2)
            return F.interpolate(seq_logits, size=x.shape[2], mode="linear", align_corners=False).squeeze(1)

# 4. Instantiate models, move to DEVICE, load state dicts, set .eval()
model_resnet_seq = ResNetSequenceClassifier().to(DEVICE)
model_resnet_seq.load_state_dict(
    torch.load(MODELS_DIR / "resnet18_physics_composite.pth", map_location=DEVICE, weights_only=False)
)
model_resnet_seq.eval()

model_resnet_std = ResNetV1Standard().to(DEVICE)
model_resnet_std.load_state_dict(
    torch.load(MODELS_DIR / "best_resnet18.pth", map_location=DEVICE, weights_only=False)
)
model_resnet_std.eval()

model_effnet_seq = EfficientNetSequenceClassifier().to(DEVICE)
model_effnet_seq.load_state_dict(
    torch.load(MODELS_DIR / "efficientnet_b0_physics.pth", map_location=DEVICE, weights_only=False)
)
model_effnet_seq.eval()

print(f"Device: {DEVICE} | Triple-Ensemble successfully initialized from {MODELS_DIR}")

Device: cpu | Triple-Ensemble successfully initialized from D:\Desktop\ABC27\models


## Cell 2: Inference Engine with Fine TTA

In [2]:
def compute_svd_energy_envelope(
    cir_rf: np.ndarray,
    k_components: int = 2,
    block_frames: int = 400,
    block_hop: int = 200,
) -> np.ndarray:
    """Calculate total kinetic energy envelope E(t) via Gram-matrix SVD clutter removal.

    For continuous, stitched 30-minute streams (~88,500 frames), overlapped 8.0s (400-frame)
    Hann-weighted blocks ensure static multipath room reflections and stationary post-fall
    body clutter are cleanly subtracted via the top-k Gram eigenvectors without causal EMA lag.
    """
    n_radars, n_frames, _ = cir_rf.shape
    if n_frames <= block_frames:
        starts = [0]
        win_template = np.ones(n_frames, dtype=np.float64)
    else:
        starts = list(range(0, n_frames - block_frames + 1, block_hop))
        if starts[-1] + block_frames < n_frames:
            starts.append(n_frames - block_frames)
        win_template = np.hanning(block_frames) + 0.1

    energy_acc = np.zeros(n_frames, dtype=np.float64)
    weight_acc = np.zeros(n_frames, dtype=np.float64)

    for s in starts:
        e = min(n_frames, s + block_frames)
        seg_len = e - s
        block_energy = np.zeros(seg_len, dtype=np.float64)

        for r in range(n_radars):
            rtm = cir_rf[r, s:e].astype(np.float64, copy=False)
            k_eff = min(k_components, min(rtm.shape) - 1)
            if seg_len >= 4 and k_eff > 0:
                gram = rtm.T @ rtm
                _, eigvecs = np.linalg.eigh(gram)
                top_k_vecs = eigvecs[:, -k_eff:]
                rtm_clean = rtm - (rtm @ top_k_vecs) @ top_k_vecs.T
            else:
                rtm_clean = rtm - rtm.mean(axis=0, keepdims=True)
            block_energy += np.sqrt(np.sum(rtm_clean * rtm_clean, axis=1))

        w = win_template[:seg_len]
        energy_acc[s:e] += block_energy * w
        weight_acc[s:e] += w

    return (energy_acc / np.maximum(weight_acc, 1e-8)).astype(np.float32)


def infer_stream_tta(
    model: nn.Module,
    cir_clean: np.ndarray,
    batch_size: int = 64,
    is_sequence: bool = True,
) -> np.ndarray:
    """Process cleaned UWB stream using sliding windows with fine stride (HOP_FRAMES = 5)."""
    # Per-radar Z-score normalization over range bins [0 : N_BINS]
    amp = np.abs(cir_clean[:, :, :N_BINS])
    mean = amp.mean(axis=(1, 2), keepdims=True)
    std = amp.std(axis=(1, 2), keepdims=True) + 1e-6
    amp_norm = ((amp - mean) / std).astype(np.float32)

    n_frames = amp_norm.shape[1]
    if n_frames <= WINDOW_FRAMES:
        starts = [0]
    else:
        starts = list(range(0, n_frames - WINDOW_FRAMES + 1, HOP_FRAMES))
        if starts[-1] != n_frames - WINDOW_FRAMES:
            starts.append(n_frames - WINDOW_FRAMES)

    prob_frames = np.zeros(n_frames, dtype=np.float64)
    prob_counts = np.zeros(n_frames, dtype=np.float64)

    model.eval()
    with torch.no_grad():
        for i in range(0, len(starts), batch_size):
            batch_starts = starts[i : i + batch_size]
            windows = np.stack(
                [amp_norm[:, s : s + WINDOW_FRAMES, :] for s in batch_starts],
                axis=0,
            )
            batch_x = torch.from_numpy(windows).to(DEVICE, dtype=torch.float32)
            outputs = model(batch_x)

            if is_sequence:
                if outputs.ndim == 3:
                    outputs = outputs.squeeze(1) if outputs.shape[1] == 1 else outputs.squeeze(-1)
                if outputs.shape[-1] != WINDOW_FRAMES:
                    outputs = F.interpolate(
                        outputs.unsqueeze(1),
                        size=WINDOW_FRAMES,
                        mode="linear",
                        align_corners=False,
                    ).squeeze(1)
                if bool((outputs.min() < 0.0) or (outputs.max() > 1.0)):
                    probs = torch.sigmoid(outputs)
                else:
                    probs = outputs
                probs_np = probs.detach().cpu().numpy()
                for j, s in enumerate(batch_starts):
                    end = min(n_frames, s + WINDOW_FRAMES)
                    prob_frames[s:end] += probs_np[j, : end - s]
                    prob_counts[s:end] += 1.0
            else:
                if outputs.ndim == 2 and outputs.shape[-1] == 2:
                    probs = torch.softmax(outputs, dim=-1)[:, 1]
                else:
                    outputs_1d = outputs.view(-1)
                    probs = (
                        torch.sigmoid(outputs_1d)
                        if bool((outputs_1d.min() < 0.0) or (outputs_1d.max() > 1.0))
                        else outputs_1d
                    )
                probs_np = probs.detach().cpu().numpy()
                for j, s in enumerate(batch_starts):
                    end = min(n_frames, s + WINDOW_FRAMES)
                    prob_frames[s:end] += float(probs_np[j])
                    prob_counts[s:end] += 1.0

    return (prob_frames / np.maximum(prob_counts, 1.0)).astype(np.float32)

## Cell 3: Dynamic Biomechanical Refinement

In [3]:
def dynamic_biomechanical_refinement(
    stream_id: str,
    ensemble_prob: np.ndarray,
    energy_envelope: np.ndarray,
    prob_threshold: float = 0.60,
) -> list:
    """Refine fall boundaries [t_start, t_end] using SVD energy E(t) and 1st derivative dE/dt."""
    n_frames = len(ensemble_prob)
    stream_dur_s = n_frames / FS_HZ

    # Continuous regions where ensemble_prob > prob_threshold
    above = np.flatnonzero(ensemble_prob > prob_threshold)
    if len(above) == 0:
        return []

    # Merge brief intra-event probability dips (< 0.45s) into contiguous candidate regions
    gap_frames = int(round(0.45 * FS_HZ))
    cuts = np.flatnonzero(np.diff(above) > gap_frames)
    region_starts = np.concatenate([[above[0]], above[cuts + 1]])
    region_ends = np.concatenate([above[cuts] + 1, [above[-1] + 1]])

    # Smooth the energy envelope and calculate its 1st derivative (dE/dt)
    smooth_e = scipy.ndimage.gaussian_filter1d(
        energy_envelope.astype(np.float64),
        sigma=0.14 * FS_HZ,
        mode="nearest",
    )
    dE_dt = np.gradient(smooth_e, 1.0 / FS_HZ)

    min_region_frames = int(round(1.20 * FS_HZ))
    min_fwhm_frames = int(round(0.68 * FS_HZ))
    min_peak_prob = max(prob_threshold, 0.675)

    refined_events = []
    for rs, re in zip(region_starts, region_ends):
        if (re - rs) < min_region_frames:
            continue

        # Find global peak t_peak within candidate region
        peak_idx = int(rs + np.argmax(ensemble_prob[rs:re]))
        score = float(ensemble_prob[peak_idx])
        if score < min_peak_prob:
            continue
        t_peak = peak_idx / FS_HZ

        # Search window around candidate region compensating for causal 3.0s EMA lag
        b_start = max(0, rs - int(0.65 * FS_HZ))
        b_end = min(n_frames - 1, max(peak_idx + int(0.35 * FS_HZ), re - int(0.20 * FS_HZ)))

        ctx_s = max(0, b_start - int(2.5 * FS_HZ))
        ctx_e = min(n_frames, re + int(2.5 * FS_HZ))
        local_base = float(np.percentile(smooth_e[ctx_s:ctx_e], 20.0))

        e_pk_idx = int(b_start + np.argmax(smooth_e[b_start : b_end + 1]))
        span = max(float(smooth_e[e_pk_idx]) - local_base, 1e-8)

        # Reject sub-biomechanical 0.2s crossfade splice artifacts via kinetic FWHM
        fwhm_frames = int(np.sum(smooth_e[b_start : max(b_end + 1, re)] > (local_base + 0.50 * span)))
        if fwhm_frames < min_fwhm_frames:
            continue

        # 1. Backward search from t_peak using dE/dt for the first major positive spike to set t_start
        search_hi = max(b_start + 3, max(peak_idx, e_pk_idx) + 1)
        grad_seg = dE_dt[b_start:search_hi]
        max_grad = float(np.max(grad_seg)) if len(grad_seg) > 0 else 0.0

        start_idx = max(0, rs - int(0.15 * FS_HZ))
        if max_grad > 1e-8 and len(grad_seg) >= 3:
            spikes, _ = scipy.signal.find_peaks(grad_seg, height=0.20 * max_grad, distance=4)
            valid_spikes = [
                b_start + int(sp)
                for sp in spikes
                if float(np.max(smooth_e[b_start + int(sp) : min(n_frames, b_start + int(sp) + 15)]))
                >= local_base + 0.16 * span
            ]
            if valid_spikes:
                sp0 = valid_spikes[0]
                while sp0 > b_start and dE_dt[sp0 - 1] > 0.05 * max_grad:
                    sp0 -= 1
                start_idx = max(0, sp0 - 1)

        # 2. Forward search from t_peak until E(t) stabilizes near the local baseline to set t_end
        f_lo = min(e_pk_idx, peak_idx)
        f_hi = min(n_frames, max(b_end + 1, f_lo + int(0.65 * FS_HZ)))
        stab_thresh = local_base + 0.60 * span
        active_rel = np.flatnonzero(smooth_e[f_lo:f_hi] > stab_thresh)
        if len(active_rel) > 0:
            end_idx = min(n_frames - 1, f_lo + int(active_rel[-1]) + 2)
        else:
            end_idx = min(n_frames - 1, peak_idx + int(0.55 * FS_HZ))

        t_start = max(0.0, start_idx / FS_HZ)
        t_end = min(stream_dur_s, end_idx / FS_HZ)

        # Minimum biomechanical fall duration guard (1.40s)
        if (t_end - t_start) < 1.40:
            deficit = 1.40 - (t_end - t_start)
            t_start = max(0.0, t_start - 0.30 * deficit)
            t_end = min(stream_dur_s, t_start + 1.40)

        # Biomechanical failsafe: if t_end - t_start > 3.5s, bound to t_peak +/- 1.2s
        if (t_end - t_start) > 3.5:
            t_start = max(0.0, t_peak - 1.2)
            t_end = min(stream_dur_s, t_peak + 1.2)

        refined_events.append([stream_id, round(t_start, 3), round(t_end, 3), round(score, 6)])

    return refined_events

## Cell 4: Evaluation and Prediction Generation

In [4]:
def run_ensemble_on_stream(h5_path: Path, stream_id: str, batch_size: int = 64) -> list:
    """Load HDF5 UWB stream, run fine-TTA Triple-Ensemble, and dynamically refine fall events."""
    with h5py.File(h5_path, "r") as handle:
        cir_raw = handle["cir"][:]

    energy_env = compute_svd_energy_envelope(cir_raw, k_components=2)
    cir_clean = subtract_background(cir_raw, 3.0)

    p1 = infer_stream_tta(model_resnet_seq, cir_clean, batch_size=batch_size, is_sequence=True)
    p2 = infer_stream_tta(model_resnet_std, cir_clean, batch_size=batch_size, is_sequence=False)
    p3 = infer_stream_tta(model_effnet_seq, cir_clean, batch_size=batch_size, is_sequence=True)

    ensemble_prob = 0.45 * p1 + 0.40 * p2 + 0.15 * p3
    return dynamic_biomechanical_refinement(
        stream_id=stream_id,
        ensemble_prob=ensemble_prob,
        energy_envelope=energy_env,
        prob_threshold=0.60,
    )


# 1. Validation Set Evaluation (val_00 to val_04)
val_streams_dir = TRAIN_PKG / "streams" / "val"
val_durations = stream_durations(val_streams_dir)
gt_val = [
    ev
    for ev in load_ground_truth(TRAIN_PKG / "streams_events.csv", ignore_flagged=True)
    if ev["stream_id"] in val_durations
]

val_events = []
for idx in range(5):
    stream_id = f"val_{idx:02d}"
    h5_file = val_streams_dir / f"stream_{idx:02d}.h5"
    events_i = run_ensemble_on_stream(h5_file, stream_id, batch_size=64)
    val_events.extend(events_i)
    print(f"[{stream_id}] Refined {len(events_i):2d} fall events (duration={val_durations[stream_id]:.2f}s)")

val_pred_records = [
    {"stream_id": r[0], "t_start": r[1], "t_end": r[2], "score": r[3], "t_alarm": None}
    for r in val_events
]
val_report = evaluate(gt_val, val_pred_records, val_durations)
rk = val_report["ranking_score"]
bt = val_report["by_theta"]

print("\n=== Official Validation Evaluation (ignore_flagged=True) ===")
print(f"Validation Ranking F1 (mean @ 0.1, 0.3, 0.5) : {rk['f1_mean']:.4f}")
print(f"  F1 @ theta=0.1                             : {bt['0.1']['f1']:.4f} (TP={bt['0.1']['tp']}, FP={bt['0.1']['fp']}, FN={bt['0.1']['fn']})")
print(f"  F1 @ theta=0.3 (Primary)                   : {bt['0.3']['f1']:.4f} (TP={bt['0.3']['tp']}, FP={bt['0.3']['fp']}, FN={bt['0.3']['fn']})")
print(f"  F1 @ theta=0.5                             : {bt['0.5']['f1']:.4f} (TP={bt['0.5']['tp']}, FP={bt['0.5']['fp']}, FN={bt['0.5']['fn']})")
print(f"  Median |t_start error| @ 0.3               : {rk['start_error_median_abs']:.3f} s")
print(f"  False Alarms / Hour @ 0.3                  : {rk['false_alarms_per_hour']:.2f}")

# 2. Test Set Prediction Generation (test_00 to test_04)
test_streams_dir = TEST_PKG / "streams" / "test"
test_durations = stream_durations(test_streams_dir)
test_events = []

print("\n=== Test Set Prediction Generation ===")
for idx in range(5):
    stream_id = f"test_{idx:02d}"
    h5_file = test_streams_dir / f"stream_{idx:02d}.h5"
    events_i = run_ensemble_on_stream(h5_file, stream_id, batch_size=64)
    test_events.extend(events_i)
    print(f"[{stream_id}] Generated {len(events_i):2d} events (duration={test_durations[stream_id]:.2f}s)")

submission_path = PROJECT_ROOT / "submission_dynamic_ensemble.csv"
submission_df = pd.DataFrame(test_events, columns=["stream_id", "t_start", "t_end", "score"])
submission_df["t_alarm"] = submission_df["t_start"]
submission_df.to_csv(submission_path, index=False)

validated_records = validate_predictions(submission_path, test_durations)
durs_s = (submission_df["t_end"] - submission_df["t_start"]).to_numpy()
print(f"\nSaved {len(validated_records)} validated predictions -> {submission_path}")
print(
    f"Event duration summary: mean={durs_s.mean():.2f}s, median={np.median(durs_s):.2f}s, "
    f"min={durs_s.min():.2f}s, max={durs_s.max():.2f}s"
)